In [6]:
# Install the required packages, including psycopg2 for PostgreSQL
%pip install -qU "sqlalchemy>=2" psycopg2-binary langchain-core langchain-community langchain-ollama langgraph python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

load_dotenv()

username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

database_uri = f"postgresql://{username}:{password}@{host}:{port}/{database}"

try:
    sqldb = SQLDatabase.from_uri(database_uri, schema="exp_v3")
    
    print("Successfully connected to the Football Database!")
    
    #print out the available tables
    names = sqldb.get_usable_table_names()
    print(f"Usable Tables ({len(names)}): {names}")
    
except Exception as e:
    print(f"Connection failed. Error: {e}")

Successfully connected to the Football Database!
Usable Tables (15): ['club', 'club_league_history', 'coach', 'coach_club_team', 'league', 'match_fact', 'national_opponent_team', 'national_team', 'player', 'player_club_team', 'player_fact', 'plays_match', 'stadium', 'world_cup', 'world_cup_result']


In [9]:
schema_snippet = sqldb.get_table_info()
print("\n--- Schema Snippet (first 1000 chars) ---")
print(schema_snippet[:1000])


--- Schema Snippet (first 1000 chars) ---

CREATE TABLE exp_v3.club (
	club_id VARCHAR NOT NULL, 
	club_name VARCHAR NOT NULL, 
	country VARCHAR, 
	found_year INTEGER, 
	CONSTRAINT club_pk PRIMARY KEY (club_id)
)

/*
3 rows from club table:
club_id	club_name	country	found_year
Q18741	Tottenham Hotspur F.C.	United Kingdom	1882
Q18708	Fulham F.C.	United Kingdom	1879
Q301973	Futbol Club Andorra Veterans	Andorra	1996
*/


CREATE TABLE exp_v3.club_league_history (
	club_id VARCHAR, 
	league_id VARCHAR, 
	start_year INTEGER, 
	end_year INTEGER, 
	CONSTRAINT club_league_history_club_id_fk FOREIGN KEY(club_id) REFERENCES exp_v3.club (club_id), 
	CONSTRAINT club_league_history_league_id_fk FOREIGN KEY(league_id) REFERENCES exp_v3.league (league_id)
)

/*
3 rows from club_league_history table:
club_id	league_id	start_year	end_year
Q1124841	Q12837728	2001	0
Q153539	Q82595	2021	2022
Q56734527	Q237181	2021	2022
*/


CREATE TABLE exp_v3.coach (
	nickname VARCHAR, 
	coach_id INTEGER NOT NULL, 
	coun